# Machine Learning Modeling Pipeline

This notebook trains, evaluates, and validates predictive machine learning models for the three psychological condition targets: **Stress (PSS-10)**, **Anxiety (GAD-7)**, and **Depression (PHQ-9)**.

### Pipeline Workflow:
1. **Step 1 — Load Data**: Load engineered feature matrix $X$ and target vectors ($y_{stress}$, $y_{anxiety}$, $y_{depression}$).
2. **Step 2 — Train/Test Split**: 70/30 stratified train/test split (`random_state=42`), maintaining index alignment across all targets.
3. **Step 3 — Logistic Regression (Baseline)**: Scaled feature matrix, `class_weight='balanced'`.
4. **Step 4 — Random Forest**: Tree-based model on unscaled features, `n_estimators=200`, `class_weight='balanced'`.
5. **Step 5 — XGBoost**: Gradient boosted decision trees on encoded integer labels, `n_estimators=200`.
6. **Step 6 — Weighted F1 Summary Table**: Comparative performance evaluation table highlighting top models.
7. **Step 7 — Stratified 5-Fold Cross Validation**: 5-fold CV evaluation on full dataset for top models.
8. **Step 8 — Save Best Models**: Export serialized model artifacts (`best_model_*.pkl`) via `joblib`.

## Step 1 — Load Data

**Rationale:**
Load engineered feature matrix `X_features.csv` (32 features) and target label files (`y_stress.csv`, `y_anxiety.csv`, `y_depression.csv`). Squeeze target dataframes into 1D Pandas Series.

**Action:** Read CSV files, flatten target Series, and verify matrix dimensions.

In [ ]:
import os
import pandas as pd
import numpy as np

# Load feature matrix and targets
X = pd.read_csv('X_features.csv')
y_stress = pd.read_csv('y_stress.csv').squeeze('columns')
y_anxiety = pd.read_csv('y_anxiety.csv').squeeze('columns')
y_depression = pd.read_csv('y_depression.csv').squeeze('columns')

print("=== DATASET LOADED SUCCESSFULLY ===")
print(f"X_features shape : {X.shape[0]} rows, {X.shape[1]} features")
print(f"y_stress shape   : {y_stress.shape[0]} samples")
print(f"y_anxiety shape  : {y_anxiety.shape[0]} samples")
print(f"y_depression shape: {y_depression.shape[0]} samples")

## Step 2 — Stratified Train/Test Split

**Rationale:**
Perform a 70% train / 30% test split (`random_state=42`) stratified on the dataset. To ensure complete index alignment across all three condition targets, the single feature partition (`X_train`, `X_test`) is generated once and used to partition `y_stress`, `y_anxiety`, and `y_depression` via matching row indices.

**Action:** Partition dataset using aligned indices and verify class distributions.

In [ ]:
from sklearn.model_selection import train_test_split

# Single 70/30 split on X and y_stress to establish consistent train/test indices
X_train, X_test, y_stress_train, y_stress_test = train_test_split(
    X, y_stress, test_size=0.30, random_state=42, stratify=y_stress
)

# Slice y_anxiety and y_depression using identical row indices
y_anxiety_train = y_anxiety.loc[X_train.index]
y_anxiety_test = y_anxiety.loc[X_test.index]

y_depression_train = y_depression.loc[X_train.index]
y_depression_test = y_depression.loc[X_test.index]

print(f"Training set shape: {X_train.shape[0]} samples")
print(f"Test set shape    : {X_test.shape[0]} samples")
print(f"Index alignment check: (X_train.index == y_anxiety_train.index) -> {(X_train.index == y_anxiety_train.index).all()}\n")

def print_stratification(name, y_tr, y_te):
    print(f"--- {name} Class Proportions (%) ---")
    df_strat = pd.DataFrame({
        'Train %': y_tr.value_counts(normalize=True) * 100,
        'Test %': y_te.value_counts(normalize=True) * 100
    })
    print(df_strat.round(2))
    print()

print_stratification("Stress Target", y_stress_train, y_stress_test)
print_stratification("Anxiety Target", y_anxiety_train, y_anxiety_test)
print_stratification("Depression Target", y_depression_train, y_depression_test)

## Step 3 — Logistic Regression (Baseline)

**Rationale:**
- Logistic Regression serves as a linear baseline model.
- `StandardScaler` is fitted on `X_train` ONLY and applied to `X_test` to prevent data leakage.
- `class_weight='balanced'` adjusts weights inversely proportional to class frequencies to handle target imbalance.

**Action:** Train Logistic Regression models for each target and report classification metrics.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Dictionary to store weighted F1 results
f1_results = {'Logistic Regression': {}, 'Random Forest': {}, 'XGBoost': {}}

# 1. Stress - Logistic Regression
lr_stress = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_stress.fit(X_train_scaled, y_stress_train)
y_pred_lr_stress = lr_stress.predict(X_test_scaled)
f1_lr_stress = f1_score(y_stress_test, y_pred_lr_stress, average='weighted')
f1_results['Logistic Regression']['Stress'] = f1_lr_stress

print("=== STRESS - LOGISTIC REGRESSION ===")
print(classification_report(y_stress_test, y_pred_lr_stress, digits=4))
print(f"Weighted F1 Score: {f1_lr_stress:.4f}\n")

# 2. Anxiety - Logistic Regression
lr_anxiety = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_anxiety.fit(X_train_scaled, y_anxiety_train)
y_pred_lr_anxiety = lr_anxiety.predict(X_test_scaled)
f1_lr_anxiety = f1_score(y_anxiety_test, y_pred_lr_anxiety, average='weighted')
f1_results['Logistic Regression']['Anxiety'] = f1_lr_anxiety

print("=== ANXIETY - LOGISTIC REGRESSION ===")
print(classification_report(y_anxiety_test, y_pred_lr_anxiety, digits=4))
print(f"Weighted F1 Score: {f1_lr_anxiety:.4f}\n")

# 3. Depression - Logistic Regression
lr_depression = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_depression.fit(X_train_scaled, y_depression_train)
y_pred_lr_depression = lr_depression.predict(X_test_scaled)
f1_lr_depression = f1_score(y_depression_test, y_pred_lr_depression, average='weighted')
f1_results['Logistic Regression']['Depression'] = f1_lr_depression

print("=== DEPRESSION - LOGISTIC REGRESSION ===")
print(classification_report(y_depression_test, y_pred_lr_depression, digits=4))
print(f"Weighted F1 Score: {f1_lr_depression:.4f}")

## Step 4 — Random Forest Classifier

**Rationale:**
- Random Forest is an ensemble tree algorithm capable of capturing complex non-linear feature interactions.
- Uses raw unscaled features (`X_train`, `X_test`).
- Configured with `n_estimators=200`, `class_weight='balanced'`, `random_state=42`.

**Action:** Train Random Forest classifiers and report classification performance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. Stress - Random Forest
rf_stress = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf_stress.fit(X_train, y_stress_train)
y_pred_rf_stress = rf_stress.predict(X_test)
f1_rf_stress = f1_score(y_stress_test, y_pred_rf_stress, average='weighted')
f1_results['Random Forest']['Stress'] = f1_rf_stress

print("=== STRESS - RANDOM FOREST ===")
print(classification_report(y_stress_test, y_pred_rf_stress, digits=4))
print(f"Weighted F1 Score: {f1_rf_stress:.4f}\n")

# 2. Anxiety - Random Forest
rf_anxiety = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf_anxiety.fit(X_train, y_anxiety_train)
y_pred_rf_anxiety = rf_anxiety.predict(X_test)
f1_rf_anxiety = f1_score(y_anxiety_test, y_pred_rf_anxiety, average='weighted')
f1_results['Random Forest']['Anxiety'] = f1_rf_anxiety

print("=== ANXIETY - RANDOM FOREST ===")
print(classification_report(y_anxiety_test, y_pred_rf_anxiety, digits=4))
print(f"Weighted F1 Score: {f1_rf_anxiety:.4f}\n")

# 3. Depression - Random Forest
rf_depression = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf_depression.fit(X_train, y_depression_train)
y_pred_rf_depression = rf_depression.predict(X_test)
f1_rf_depression = f1_score(y_depression_test, y_pred_rf_depression, average='weighted')
f1_results['Random Forest']['Depression'] = f1_rf_depression

print("=== DEPRESSION - RANDOM FOREST ===")
print(classification_report(y_depression_test, y_pred_rf_depression, digits=4))
print(f"Weighted F1 Score: {f1_rf_depression:.4f}")

## Step 5 — XGBoost Classifier

**Rationale:**
- XGBoost requires integer target encoding (0 to $K-1$). `LabelEncoder` is fitted on training labels and transforms test labels.
- Predictions are inverse-transformed back to original label strings for evaluation.
- Configured with `n_estimators=200`, `eval_metric='mlogloss'`, `random_state=42`.

**Action:** Encode labels, train XGBoost classifiers, inverse transform predictions, and print metrics.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# 1. Stress - XGBoost
le_stress = LabelEncoder()
y_stress_tr_enc = le_stress.fit_transform(y_stress_train)
y_stress_te_enc = le_stress.transform(y_stress_test)

xgb_stress = XGBClassifier(n_estimators=200, eval_metric='mlogloss', random_state=42)
xgb_stress.fit(X_train, y_stress_tr_enc)
y_pred_xgb_stress_enc = xgb_stress.predict(X_test)
y_pred_xgb_stress = le_stress.inverse_transform(y_pred_xgb_stress_enc)
f1_xgb_stress = f1_score(y_stress_test, y_pred_xgb_stress, average='weighted')
f1_results['XGBoost']['Stress'] = f1_xgb_stress

print("=== STRESS - XGBOOST ===")
print(classification_report(y_stress_test, y_pred_xgb_stress, digits=4))
print(f"Weighted F1 Score: {f1_xgb_stress:.4f}\n")

# 2. Anxiety - XGBoost
le_anxiety = LabelEncoder()
y_anxiety_tr_enc = le_anxiety.fit_transform(y_anxiety_train)
y_anxiety_te_enc = le_anxiety.transform(y_anxiety_test)

xgb_anxiety = XGBClassifier(n_estimators=200, eval_metric='mlogloss', random_state=42)
xgb_anxiety.fit(X_train, y_anxiety_tr_enc)
y_pred_xgb_anxiety_enc = xgb_anxiety.predict(X_test)
y_pred_xgb_anxiety = le_anxiety.inverse_transform(y_pred_xgb_anxiety_enc)
f1_xgb_anxiety = f1_score(y_anxiety_test, y_pred_xgb_anxiety, average='weighted')
f1_results['XGBoost']['Anxiety'] = f1_xgb_anxiety

print("=== ANXIETY - XGBOOST ===")
print(classification_report(y_anxiety_test, y_pred_xgb_anxiety, digits=4))
print(f"Weighted F1 Score: {f1_xgb_anxiety:.4f}\n")

# 3. Depression - XGBoost
le_depression = LabelEncoder()
y_depression_tr_enc = le_depression.fit_transform(y_depression_train)
y_depression_te_enc = le_depression.transform(y_depression_test)

xgb_depression = XGBClassifier(n_estimators=200, eval_metric='mlogloss', random_state=42)
xgb_depression.fit(X_train, y_depression_tr_enc)
y_pred_xgb_depression_enc = xgb_depression.predict(X_test)
y_pred_xgb_depression = le_depression.inverse_transform(y_pred_xgb_depression_enc)
f1_xgb_depression = f1_score(y_depression_test, y_pred_xgb_depression, average='weighted')
f1_results['XGBoost']['Depression'] = f1_xgb_depression

print("=== DEPRESSION - XGBOOST ===")
print(classification_report(y_depression_test, y_pred_xgb_depression, digits=4))
print(f"Weighted F1 Score: {f1_xgb_depression:.4f}")

## Step 6 — Weighted F1 Summary Table

**Rationale:**
Construct a comparative summary matrix showing weighted F1 performance across all three algorithms for each target condition to identify top-performing models.

**Action:** Format and display summary table, highlighting winning models.

In [ ]:
summary_df = pd.DataFrame(f1_results)
summary_df = summary_df[['Logistic Regression', 'Random Forest', 'XGBoost']]

print("=" * 70)
print("              WEIGHTED F1 SCORE SUMMARY TABLE")
print("=" * 70)
print(summary_df.round(4).to_string())
print("-" * 70)

best_models = {}
for target in ['Stress', 'Anxiety', 'Depression']:
    best_model_name = summary_df.loc[target].idxmax()
    best_f1 = summary_df.loc[target].max()
    best_models[target] = best_model_name
    print(f"* Best Model for {target:<10}: {best_model_name} (Weighted F1: {best_f1:.4f})")
print("=" * 70)

## Step 7 — Stratified K-Fold Cross Validation

**Rationale:**
- Evaluates the best model per target using 5-fold stratified cross-validation on the FULL dataset `X` and target `y`.
- Uses `f1_weighted` as metric to report mean and standard deviation.
- For Logistic Regression, a pipeline incorporating `StandardScaler` is used to prevent data leakage during CV folds.

**Action:** Run 5-fold CV for best models and print performance estimates.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def get_cv_estimator(model_name):
    if model_name == 'Logistic Regression':
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    elif model_name == 'Random Forest':
        return RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
    elif model_name == 'XGBoost':
        return XGBClassifier(n_estimators=200, eval_metric='mlogloss', random_state=42)

# 1. Cross Validation for Stress
best_stress_name = best_models['Stress']
est_stress = get_cv_estimator(best_stress_name)
if best_stress_name == 'XGBoost':
    y_stress_cv = LabelEncoder().fit_transform(y_stress)
else:
    y_stress_cv = y_stress

scores_stress = cross_val_score(est_stress, X, y_stress_cv, cv=cv, scoring='f1_weighted')
print(f"=== CROSS VALIDATION - STRESS (Best Model: {best_stress_name}) ===")
print(f"5-Fold Weighted F1 Scores: {[round(s, 4) for s in scores_stress]}")
print(f"Mean Weighted F1: {scores_stress.mean():.4f} +/- {scores_stress.std():.4f}\n")

# 2. Cross Validation for Anxiety
best_anxiety_name = best_models['Anxiety']
est_anxiety = get_cv_estimator(best_anxiety_name)
if best_anxiety_name == 'XGBoost':
    y_anxiety_cv = LabelEncoder().fit_transform(y_anxiety)
else:
    y_anxiety_cv = y_anxiety

scores_anxiety = cross_val_score(est_anxiety, X, y_anxiety_cv, cv=cv, scoring='f1_weighted')
print(f"=== CROSS VALIDATION - ANXIETY (Best Model: {best_anxiety_name}) ===")
print(f"5-Fold Weighted F1 Scores: {[round(s, 4) for s in scores_anxiety]}")
print(f"Mean Weighted F1: {scores_anxiety.mean():.4f} +/- {scores_anxiety.std():.4f}\n")

# 3. Cross Validation for Depression
best_dep_name = best_models['Depression']
est_dep = get_cv_estimator(best_dep_name)
if best_dep_name == 'XGBoost':
    y_dep_cv = LabelEncoder().fit_transform(y_depression)
else:
    y_dep_cv = y_depression

scores_dep = cross_val_score(est_dep, X, y_dep_cv, cv=cv, scoring='f1_weighted')
print(f"=== CROSS VALIDATION - DEPRESSION (Best Model: {best_dep_name}) ===")
print(f"5-Fold Weighted F1 Scores: {[round(s, 4) for s in scores_dep]}")
print(f"Mean Weighted F1: {scores_dep.mean():.4f} +/- {scores_dep.std():.4f}")

## Step 8 — Save Best Models

**Rationale:**
Serialize top-performing models using `joblib` for deployment in the chatbot inference engine. If Logistic Regression is a top model, save `scaler.pkl` to scale future user input features.

**Action:** Export serialized model artifacts (`best_model_stress.pkl`, `best_model_anxiety.pkl`, `best_model_depression.pkl`) and optional `scaler.pkl`.

In [ ]:
import joblib

# Dictionary of trained models mapping
model_objects = {
    'Stress': {'Logistic Regression': lr_stress, 'Random Forest': rf_stress, 'XGBoost': xgb_stress},
    'Anxiety': {'Logistic Regression': lr_anxiety, 'Random Forest': rf_anxiety, 'XGBoost': xgb_anxiety},
    'Depression': {'Logistic Regression': lr_depression, 'Random Forest': rf_depression, 'XGBoost': xgb_depression}
}

print("=== SAVING BEST MODEL ARTIFACTS ===")
needs_scaler = False

for target, fname in [('Stress', 'best_model_stress.pkl'), ('Anxiety', 'best_model_anxiety.pkl'), ('Depression', 'best_model_depression.pkl')]:
    best_name = best_models[target]
    best_obj = model_objects[target][best_name]
    joblib.dump(best_obj, fname)
    print(f"[OK] Saved '{fname}' ({target} -> {best_name})")
    if best_name == 'Logistic Regression':
        needs_scaler = True

if needs_scaler:
    joblib.dump(scaler, 'scaler.pkl')
    print("[OK] Saved 'scaler.pkl' (Fitted StandardScaler for Logistic Regression)")
else:
    joblib.dump(scaler, 'scaler.pkl')
    print("[OK] Saved 'scaler.pkl' (StandardScaler saved for deployment pipeline)")

print("\nAll model artifacts successfully exported!")